Shark Attack

In [1]:
import pandas as pd
%pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [2]:
shark = pd.read_excel('GSAF5.xls')
shark.columns = shark.columns.str.strip()

In [18]:
sex_mapping = {
    "M": "male",
    " M ":"male",
    "M ": "male",
    "m": "male",
    "M x 2": "male",
    "F": "female",
    "F ": "female",
    "M F": "not defined",
    "?": "not defined",
    "N": "not defined",
    "lli": "not defined",
    ".": "not defined"
}


keywords = {
    "spearfish": "Fishing",
    "spear fish": "Fishing",
    "fishing": "Fishing",
    "fisherman": "Fishing",
    "fishermen": "Fishing",
    "fish": "Fishing",

    "swimming": "Swimming",
    "swimmer": "Swimming",
    "swimmers": "Swimming",
    "swam": "Swimming",
    "swim": "Swimming",
    "treading water": "Swimming",
    "floating": "Swimming",

    "windsurf": "Surfing",
    "kitesurf": "Surfing",
    "kite surf": "Surfing",
    "body surf": "Surfing",
    "bodysurf": "Surfing",
    "boogie board": "Surfing",
    "bodyboard": "Surfing",
    "paddleboard": "Surfing",
    "paddle board": "Surfing",
    "surfing": "Surfing",
    "surfer": "Surfing",
    "surf": "Surfing",

    "wading": "Wading",
    "waded": "Wading",
    "wade": "Wading",
    "ankle-deep": "Wading",
    "waist-deep": "Wading",
    "knee-deep": "Wading",
    "shallow water": "Wading",

    "bathing": "Bathing",
    "bather": "Bathing",
    "bath": "Bathing",

    "scuba": "Diving",
    "free diving": "Diving",
    "freediving": "Diving",
    "skin diving": "Diving",
    "diving": "Diving",
    "diver": "Diving",
    "dive": "Diving",
    "snorkeling": "Diving",
    "snorkelling": "Diving",
    "snorkel": "Diving",

    "water skiing": "Water Skiing",
    "waterskiing": "Water Skiing",
    "water-skiing": "Water Skiing",
    "skiing": "Water Skiing",

    "kayaking": "Kayaking",
    "kayaker": "Kayaking",
    "kayak": "Kayaking",

    "canoeing": "Canoeing",
    "canoe": "Canoeing",

    "rowing": "Rowing",
    "rowboat": "Rowing",
    "rowing boat": "Rowing",

    "standing": "Standing",
    "stood": "Standing",

    "walking": "Walking",
    "walked": "Walking",
    "walking in water": "Walking",

    "playing": "Playing",
    "play": "Playing",

    "washing": "Washing",
    "cleaning": "Washing",

    "feeding shark": "Handling Shark",
    "feeding sharks": "Handling Shark",
    "carrying shark": "Handling Shark",
    "handling shark": "Handling Shark",
    "caught shark": "Handling Shark",
    "catching shark": "Handling Shark",
    "shark fishing": "Handling Shark",

    "shipwreck": "Sea Disaster",
    "wreck": "Sea Disaster",
    "capsized": "Sea Disaster",
    "capsize": "Sea Disaster",
    "sinking": "Sea Disaster",
    "sank": "Sea Disaster",

    "fell overboard": "Boating",
    "overboard": "Boating",
    "dinghy": "Boating",
    "sailing": "Boating",
    "sailboat": "Boating",
    "yacht": "Boating",
    "catamaran": "Boating",
    "boat": "Boating",
    "ship": "Boating",
    "vessel": "Boating",

    "rescuing": "Rescue",
    "rescue": "Rescue"
}


def data_cleaning(df):
    cleandf = df.copy()
    location_columns = ["Country", "State", "Location"]
    mask = cleandf[location_columns].isna().all(axis=1)
    cleandf.loc[mask, location_columns] = "Unspecified"
    cleandf = cleandf.drop(
        columns=[
            "Source", "pdf", "href formula", "href", "Case Number",
            "Case Number.1", "original order", "Unnamed: 21", "Unnamed: 22"
        ],
        errors="ignore"
    )
    cleandf = cleandf.dropna(subset=["Year"])
    cleandf["Type"] = cleandf["Type"].str.strip().str.capitalize()
    cleandf["Type"] = cleandf["Type"].where(
        cleandf["Type"].isin(["Unprovoked", "Provoked"]),
        "Unconfirmed"
    )
    
    cleandf["Age"] = cleandf["Age"].fillna("No_Age")
    cleandf["Location"] = cleandf["Location"].fillna("Unspecified")
    cleandf["Country"] = cleandf["Country"].fillna(cleandf["State"])
    cleandf["State"] = cleandf["State"].fillna(cleandf["Country"])
    cleandf[["Country", "State"]] = (
    cleandf[["Country", "State"]].fillna("Unspecified")
)
    cleandf["Species"] = cleandf["Species"].fillna("Unspecified")
    cleandf["Time"] = cleandf["Time"].fillna("Unspecified")
    cleandf["Name"] = cleandf["Name"].fillna("No_NAME")
    cleandf["Activity"] = cleandf["Activity"].fillna("Not_defined")
    for keyword, category in keywords.items():
        activity_mask = cleandf["Activity"].str.contains(
            keyword,
            case=False,
            na=False
        )
        cleandf.loc[activity_mask, "Activity"] = category
    cleandf["Fatal Y/N"] = cleandf["Fatal Y/N"].astype(str).str.strip().str.upper()
    cleandf["Fatal Y/N"] = cleandf["Fatal Y/N"].where( cleandf["Fatal Y/N"].isin(["Y", "N"]), "Unknown")
    cleandf["Sex"] = cleandf["Sex"].str.strip()
    cleandf["Sex"] = cleandf["Sex"].replace(sex_mapping)
    cleandf["Sex"] = cleandf["Sex"].fillna("not defined")
    cleandf.loc[~cleandf["Sex"].isin(["Male", "Female", "not defined"]), "sex"] = "not defined"


    
    cleandf = cleandf.drop_duplicates()
    return cleandf




In [19]:
shark_clean = data_cleaning(shark)

In [20]:
shark_clean.to_csv("shark_clean.csv", index=False)

In [23]:
shark_clean.shape

(7121, 15)

In [24]:
shark_clean["State"].value_counts()

State
Florida                              1199
New South Wales                       522
Queensland                            355
Hawaii                                347
California                            328
                                     ... 
Bikini Atoll                            1
Between New Ireland & New Britain       1
Ba Ria-Vung Tau  Province               1
Moala Island                            1
ASIA?                                   1
Name: count, Length: 1087, dtype: int64

In [ ]:
# Fatal: keep Y/N, everything else (incl. empty) → Unknown

shark_clean["Fatal Y/N"] = shark_clean["Fatal Y/N"].astype(str).str.strip().str.upper()shark_clean["Fatal Y/N"] = shark_clean["Fatal Y/N"].where( shark_clean["Fatal Y/N"].isin(["Y", "N"]), "Unknown")

# Injury: fill empty values
shark_clean["Injury"] = shark_clean["Injury"].fillna("Unknown")


Pablo Ferreira  [21:47]
shark_clean["sex"] = shark_clean["sex"].fillna("not defined")

sex_mapping = {
    "M": "male",
    "M ": "male",
    "m": "male",
    "M x 2": "male",
    "F": "female",
    "F ": "female",
    "M F": "not defined",
    "?": "not defined",
    "N": "not defined",
    "lli": "not defined",
    ".": "not defined"
}
shark_clean["sex"] = shark_clean["sex"].replace(sex_mapping)

###PRIMO SHARK_CLEAN###

In [ ]:
Activity_dic = {"Surfing": 0, ""}

In [58]:
keywords = {
    "spearfish": "Fishing",
    "spear fish": "Fishing",
    "fishing": "Fishing",
    "fisherman": "Fishing",
    "fishermen": "Fishing",
    "fish": "Fishing",

    "swimming": "Swimming",
    "swimmer": "Swimming",
    "swimmers": "Swimming",
    "swam": "Swimming",
    "swim": "Swimming",
    "treading water": "Swimming",
    "floating": "Swimming",

    "windsurf": "Surfing",
    "kitesurf": "Surfing",
    "kite surf": "Surfing",
    "body surf": "Surfing",
    "bodysurf": "Surfing",
    "boogie board": "Surfing",
    "bodyboard": "Surfing",
    "paddleboard": "Surfing",
    "paddle board": "Surfing",
    "surfing": "Surfing",
    "surfer": "Surfing",
    "surf": "Surfing",

    "wading": "Wading",
    "waded": "Wading",
    "wade": "Wading",
    "ankle-deep": "Wading",
    "waist-deep": "Wading",
    "knee-deep": "Wading",
    "shallow water": "Wading",

    "bathing": "Bathing",
    "bather": "Bathing",
    "bath": "Bathing",

    "scuba": "Diving",
    "free diving": "Diving",
    "freediving": "Diving",
    "skin diving": "Diving",
    "diving": "Diving",
    "diver": "Diving",
    "dive": "Diving",
    "snorkeling": "Diving",
    "snorkelling": "Diving",
    "snorkel": "Diving",

    "water skiing": "Water Skiing",
    "waterskiing": "Water Skiing",
    "water-skiing": "Water Skiing",
    "skiing": "Water Skiing",

    "kayaking": "Kayaking",
    "kayaker": "Kayaking",
    "kayak": "Kayaking",

    "canoeing": "Canoeing",
    "canoe": "Canoeing",

    "rowing": "Rowing",
    "rowboat": "Rowing",
    "rowing boat": "Rowing",

    "standing": "Standing",
    "stood": "Standing",

    "walking": "Walking",
    "walked": "Walking",
    "walking in water": "Walking",

    "playing": "Playing",
    "play": "Playing",

    "washing": "Washing",
    "cleaning": "Washing",

    "feeding shark": "Handling Shark",
    "feeding sharks": "Handling Shark",
    "carrying shark": "Handling Shark",
    "handling shark": "Handling Shark",
    "caught shark": "Handling Shark",
    "catching shark": "Handling Shark",
    "shark fishing": "Handling Shark",

    "shipwreck": "Sea Disaster",
    "wreck": "Sea Disaster",
    "capsized": "Sea Disaster",
    "capsize": "Sea Disaster",
    "sinking": "Sea Disaster",
    "sank": "Sea Disaster",

    "fell overboard": "Boating",
    "overboard": "Boating",
    "dinghy": "Boating",
    "sailing": "Boating",
    "sailboat": "Boating",
    "yacht": "Boating",
    "catamaran": "Boating",
    "boat": "Boating",
    "ship": "Boating",
    "vessel": "Boating",

    "rescuing": "Rescue",
    "rescue": "Rescue"
}
for keyword, category in keywords.items():
    shark_clean.loc[
        shark_clean['Activity'].str.contains(keyword, case=False, na=False),
        'Activity'
    ] = category

In [59]:
shark_clean['Activity'] = shark_clean['Activity'].fillna('not_defined')

In [60]:
shark_clean['Name'] = shark_clean['Name'].fillna('No_NAME')

In [72]:
shark_clean['Time'] = shark_clean['Time'].fillna('unspecified')

In [73]:
shark_clean['Species'] = shark_clean['Species'].fillna('unspecified')

In [61]:
shark_clean['Age'] = shark_clean['Age'].fillna('No_Age')


In [66]:
shark_clean[["Injury", "Fatal Y/N"]].isna().all(axis=1).sum()

np.int64(8)

In [67]:
location_columns = ["Country", "State", "Location"]

mask = shark_clean[location_columns].isna().all(axis=1)

shark_clean.loc[mask, location_columns] = "Unspecified"

In [68]:
shark_clean[location_columns].isna().all(axis=1).sum()

np.int64(0)

In [69]:
shark_clean[["Injury", "Fatal Y/N"]].isna().all(axis=1).sum()

np.int64(8)

In [ ]:
injury_keywords = {
    r"\bfoot\b": "Foot bitten",
    r"\bfeet\b": "Foot bitten",
    r"\btoe\b": "Foot bitten",
    r"\btoes\b": "Foot bitten",
    r"\bankle\b": "Foot bitten",

    r"\bleg\b": "Leg bitten",
    r"\blegs\b": "Leg bitten",
    r"\bcalf\b": "Leg bitten",
    r"\bthigh\b": "Leg bitten",
    r"\bknee\b": "Leg bitten",
    r"lost leg": "Leg bitten",
    r"severed leg": "Leg bitten",

    r"\bhand\b": "Hand bitten",
    r"\bhands\b": "Hand bitten",
    r"\bfinger\b": "Hand bitten",
    r"\bfingers\b": "Hand bitten",
    r"\bwrist\b": "Hand bitten",

    r"\barm\b": "Arm bitten",
    r"\barms\b": "Arm bitten",
    r"\belbow\b": "Arm bitten",
    r"\bshoulder\b": "Arm bitten",

    r"\bhead\b": "Head or face injury",
    r"\bface\b": "Head or face injury",
    r"\bneck\b": "Head or face injury",
    r"\bscalp\b": "Head or face injury",

    r"\bchest\b": "Body injury",
    r"\btorso\b": "Body injury",
    r"\babdomen\b": "Body injury",
    r"\bback\b": "Body injury",
    r"\bside\b": "Body injury",

    r"multiple bite": "Multiple injuries",
    r"multiple injur": "Multiple injuries",
    r"both legs": "Multiple injuries",
    r"both arms": "Multiple injuries",
    r"severe injuries": "Multiple injuries",

    r"\blaceration": "Minor injury",
    r"\bcut\b": "Minor injury",
    r"\bcuts\b": "Minor injury",
    r"\bscratch": "Minor injury",
    r"\babrasion": "Minor injury",
    r"\bbruise": "Minor injury",

    r"no injury": "No injury",
    r"not injured": "No injury",
    r"uninjured": "No injury",
    r"no injuries": "No injury",

    r"\bsurvived\b": "Survived",

    r"\bfatal\b": "FATAL",
    r"\bdied\b": "FATAL",
    r"\bdeath\b": "FATAL",
    r"\bkilled\b": "FATAL",
    r"body not recovered": "FATAL",
    r"body was not recovered": "FATAL",
    r"carried off by shark": "FATAL",
    r"bit him in half": "FATAL",
    r"\bdecapitated\b": "FATAL"
}

for keyword, category in keywords.items():
    shark_clean.loc[
        shark_clean['Activity'].str.contains(keyword, case=False, na=False),
        'Activity'
    ] = category

In [ ]:
x = 15 + 18